## Parte 2 – Codificação simbólico-numérica e ausentes categóricos · _responsável: Samuel Alves_
Compara como transformar `protocol_type`, `service` e `flag` em números, mantendo **fixos** a limpeza (`F_E+leakage`), a escala (`standard`), o balanceamento (`nenhum`) e o modelo. Variantes:

| Variante | O que muda |
|---|---|
| `onehot_puro` | uma coluna por categoria, sem agrupar nada |
| `onehot` (padrão da estrutura comum) | one-hot com categorias raras (< 20 linhas) agrupadas em "infrequente" |
| `ordinal` | um inteiro por categoria (ordem alfabética, arbitrária) |
| `frequency` | frequência relativa da categoria no treino do fold |
| `onehot` + `ausente` | igual ao `onehot`, mas o valor faltante vira a categoria explícita `ausente` em vez da moda |

Tudo que aprende algo (vocabulário, frequências, moda, categorias raras) é ajustado dentro do Pipeline, só no treino de cada fold. Rodamos com **RF** (modelo padrão da equipe) e **LR** (sensível a como a categoria vira número).

In [6]:
# 2.1 Diagnóstico dos atributos categóricos (após a limpeza escolhida)
CAT_COLS = [c for c in ['protocol_type', 'service', 'flag'] if c in train_clean.columns]
y_p2 = train_clean[TARGET].astype(str).str.strip().str.lower()

diag = pd.DataFrame({
    'níveis': train_clean[CAT_COLS].nunique(),
    'ausentes': train_clean[CAT_COLS].isna().sum(),
    'ausentes_%': (train_clean[CAT_COLS].isna().mean() * 100).round(2),
    'níveis_<20_linhas': [int((train_clean[c].value_counts() < 20).sum()) for c in CAT_COLS],
})
display(diag)

print('Ausência por classe (%):')
display((train_clean[CAT_COLS].isna().groupby(y_p2).mean() * 100).round(2))

serv = pd.crosstab(train_clean['service'], y_p2)
serv['total'] = serv.sum(axis=1)
serv['%_anomaly'] = (serv['anomaly'] / serv['total'] * 100).round(1)
display(serv.sort_values('total', ascending=False).head(15))

# Por que normalizar antes: espaços no fim criam categorias falsas em service
bruto = train['service'].dropna().astype(str)
print(f"service: {bruto.nunique()} níveis brutos -> {bruto.str.lower().nunique()} com lower -> "
      f"{bruto.str.strip().str.lower().nunique()} com strip+lower")

,níveis,ausentes,ausentes_%,níveis_<20_linhas
protocol_type,3,0,0.00,0
service,65,3493,6.34,42
flag,11,3225,5.86,3


Ausência por classe (%):


,protocol_type,service,flag
class,,,
anomaly,0.0,6.77,6.44
normal,0.0,6.33,5.84


class,anomaly,normal,total,%_anomaly
service,,,,
http,108,29178,29286,0.4
domain_u,1,6272,6273,0.0
smtp,12,4922,4934,0.2
ftp_data,44,3380,3424,1.3
private,607,1152,1759,34.5
other,62,1679,1741,3.6
ftp,26,830,856,3.0
telnet,72,600,672,10.7
finger,32,406,438,7.3


service: 96 níveis brutos -> 96 com lower -> 67 com strip+lower


In [7]:
# 2.2 Dimensionalidade gerada por cada codificação
ENCODERS['onehot_puro'] = lambda: OneHotEncoder(handle_unknown='ignore', sparse_output=False)

X_p2 = train_clean.drop(columns=[TARGET])
dims = []
for enc in ['onehot_puro', 'onehot', 'ordinal', 'frequency']:
    prep, n_cat = make_preprocessor(X_p2, encoder=enc)
    Xt = prep.fit_transform(X_p2)          # só descritivo: conta colunas, não avalia nada
    n_num = len(X_p2.select_dtypes('number').columns)
    dims.append({'codificacao': enc, 'colunas_categoricas': Xt.shape[1] - n_num, 'colunas_totais': Xt.shape[1]})
dims = pd.DataFrame(dims)
display(dims)

ax = dims.plot.barh(x='codificacao', y='colunas_categoricas', legend=False, figsize=(7, 3), color='#4C72B0')
ax.set(xlabel='Colunas geradas por protocol_type + service + flag', ylabel='', title='Dimensionalidade por codificação')
ax.invert_yaxis(); plt.tight_layout(); plt.savefig('parte2_dimensionalidade.png', dpi=150); plt.show()

,codificacao,colunas_categoricas,colunas_totais
0,onehot_puro,79,115
1,onehot,36,72
2,ordinal,3,39
3,frequency,3,39


In [8]:
# 2.3 Comparação por validação cruzada (mesma estrutura comum da Parte 1)
inicio_p2 = len(RESULTS)

for model in ['RF', 'LR']:
    for enc in ['onehot_puro', 'onehot', 'ordinal', 'frequency']:
        run_experiment(train_clean, CLEAN_NAME, encoder=enc, model=model)

    # Imputação: categoria explícita 'ausente' em vez da moda. fillna com constante não aprende nada
    # dos dados, então pode ser feito antes do CV sem vazamento.
    train_ausente = train_clean.copy()
    train_ausente[CAT_COLS] = train_ausente[CAT_COLS].fillna('ausente')
    run_experiment(train_ausente, CLEAN_NAME, encoder='onehot', model=model, tag='imput_ausente')

res_p2 = pd.DataFrame(RESULTS[inicio_p2:])
res_p2['imputacao_cat'] = np.where(res_p2['reducao'] == 'imput_ausente', 'ausente', 'moda')
res_p2['reducao'] = 'nenhuma'
res_p2 = res_p2.merge(dims[['codificacao', 'colunas_categoricas']], on='codificacao', how='left')
res_p2.to_csv('resultados_parte2_codificacao.csv', index=False)

cols = ['modelo', 'codificacao', 'imputacao_cat', 'colunas_categoricas', 'f1_macro', 'f1_macro_std', 'precision_macro', 'recall_macro']
res_p2.sort_values(['modelo', 'f1_macro'], ascending=[True, False])[cols].round(4)

,modelo,codificacao,imputacao_cat,colunas_categoricas,f1_macro,f1_macro_std,precision_macro,recall_macro
9,LR,onehot,ausente,36,0.9340,0.0093,0.9737,0.9010
6,LR,onehot,moda,36,0.9317,0.0036,0.9717,0.8983
5,LR,onehot_puro,moda,79,0.9309,0.0041,0.9707,0.8977
7,LR,ordinal,moda,3,0.9075,0.0040,0.9594,0.8667
8,LR,frequency,moda,3,0.9006,0.0045,0.9633,0.8537
1,RF,onehot,moda,36,0.9765,0.0035,0.9936,0.9606
4,RF,onehot,ausente,36,0.9763,0.0029,0.9941,0.9598
2,RF,ordinal,moda,3,0.9753,0.0047,0.9918,0.9600
3,RF,frequency,moda,3,0.9749,0.0035,0.9932,0.9581
0,RF,onehot_puro,moda,79,0.9740,0.0042,0.9929,0.9567


In [9]:
# 2.4 Regra de decisão (fixada antes de ver os resultados): no modelo da equipe (RF), entre as
# variantes a até 1 desvio-padrão do melhor F1-macro, escolher a de menos colunas.
rf = res_p2[res_p2['modelo'] == 'RF'].sort_values('f1_macro', ascending=False)
melhor = rf.iloc[0]
candidatas = rf[rf['f1_macro'] >= melhor['f1_macro'] - melhor['f1_macro_std']]
escolhida = candidatas.sort_values(['colunas_categoricas', 'f1_macro'], ascending=[True, False]).iloc[0]
print(f"Melhor F1-macro (RF): {melhor['codificacao']} / {melhor['imputacao_cat']} = {melhor['f1_macro']:.4f} ± {melhor['f1_macro_std']:.4f}")
print(f"Escolhida pela regra: {escolhida['codificacao']} / imputação {escolhida['imputacao_cat']}")

Melhor F1-macro (RF): onehot / moda = 0.9765 ± 0.0035
Escolhida pela regra: ordinal / imputação moda


### 2.5 Justificativa

**Diagnóstico.** Após `strip` + `lower`, `service` tem 65 níveis no treino limpo (eram 96 brutos: 29 eram o mesmo serviço com espaço no final, como `'http '`), `flag` tem 11 e `protocol_type` 3. `service` é muito concentrado: 42 níveis têm menos de 20 linhas, e alguns raros são muito informativos (`private`: 34,5% de anomalias; `ecr_i`: 32,7%; `eco_i`: 25,2%, contra 0,4% em `http`). `service` e `flag` têm cerca de 6% de ausentes, com taxa quase igual nas duas classes (6,8% × 6,3% em `service`).

**Resultados.**
- **RF:** todas as codificações empatam (F1-macro de 0,974 a 0,977, diferenças menores que 1 desvio-padrão). Árvores separam categorias por limiares, então a ordem artificial do ordinal e a perda de identidade da frequência quase não atrapalham.
- **LR:** o one-hot fica claramente acima (0,932 × 0,908 do ordinal e 0,901 da frequência). Um modelo linear dá um único peso ao código numérico, então não consegue isolar serviços raros e suspeitos como `private`, `ecr_i` e `eco_i`; o ordinal ainda impõe uma ordem alfabética sem significado.
- **Alta cardinalidade:** agrupar as categorias com menos de 20 linhas (`onehot` padrão) reduz as colunas de 79 para 36 sem perda de desempenho, nos dois modelos, em relação ao one-hot puro.
- **Imputação:** a categoria `ausente` não muda o resultado em relação à moda (diferença menor que 1 desvio-padrão), o que bate com a ausência quase igual entre as classes.
- **Vazamento:** nenhuma técnica usa o rótulo (por isso não usamos *target encoding*); vocabulário, frequências, moda e categorias raras são aprendidos só no treino de cada fold.

**Decisão.** Pela regra fixada antes de ver os resultados, com o RF (modelo da equipe) a escolha é **`ordinal` com imputação pela moda**: mesmo desempenho com 3 colunas em vez de 36 ou 79. Ela também é a codificação que o `SMOTENC` da Parte 4 exige. Para modelos lineares ou baseados em distância (LR, KNN, redes), usar **`onehot` com raras agrupadas**.